# Legged Robot State Estimation Tutorial — Teacher Version

## Overview

In this tutorial, you will build a **state estimator for a legged robot** using an **Extended Kalman Filter (EKF)**. The estimator fuses:

- **IMU data** (for motion prediction)  
- **Foot contact information** (for drift correction)

<div style="text-align: left;">
  <img src="figures/quad_coordinate_frames.png"
       alt="Quadruped robot with coordinate frames" 
       width="400">
  <p><em>Figure: Quadruped robot with coordinate frame for the floating base {B} and world frame {W}, as well as foot frames {F<sub>i</sub>}.</em></p>
</div>

---

## Learning Goals

You will:

- Implement the **propagation (IMU) step**
- Implement the **correction step using foot contacts**
- Explore the effect of **noise, bias, and modeling assumptions**
- Understand failure cases such as **drift, slip, and loss of observability**
- Apply the filter to **simulation and real data**
  

## Part A. Code Base

---

### 1. Tutorial Settings 

This section contains all configurable parameters for the tutorial, including dataset selection, EKF noise tuning, sensor models (noise, bias, slip), and visualization options.

You can explore the options, but you **do not need to modify** anything here. You will be given instructions for the tasks below.

In [4]:
"""
EKF example for a quadruped robot.

This script shows how IMU prediction and stance-foot updates interact in a
legged robot state estimation example.

Main simplifications:
- Rotation is stored as a 3x3 matrix in the state.
- Foot updates only constrain base position and foot world positions.
- IMU biases are injected optionally, but not estimated.
- Foot slip is modeled with a simple random perturbation.
"""

import os
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from scipy.linalg import expm
from scipy.spatial.transform import Rotation as R

np.random.seed(26)

# =============================================================================
# 1. Tutorial settings
# =============================================================================

# Dataset paths
SIMULATED_DATA_CSV_PATH = "datasets/quadruped_sim.csv"
REAL_DATA_CSV_PATH = "datasets/quadruped_real.csv"

# Experiment selection
USE_REAL_DATA = False

# IMU noise, bias and gravity vector
USE_IMU_NOISE = False
ACC_NOISE_STD = 0.1
GYRO_NOISE_STD = np.array([0.001, 0.001, 0.005])

USE_IMU_BIAS = False
ACC_BIAS = np.array([0.08, -0.05, 0.03])
GYRO_BIAS = np.array([0.0, 0.0, 0.02])

GRAVITY_WORLD = np.array([0.0, 0.0, -9.81])

# Foot slip
FOOT_SLIP_STD = 0 # TODO: 0.01
FOOT_SLIP_ENABLE_Z = False

# EKF noise tuning
PROCESS_NOISE_ROT = 5e-2
PROCESS_NOISE_VEL = np.array([7e-2, 3e-2, 2e-2])
PROCESS_NOISE_POS = np.array([7e-2, 3e-2, 2e-2])

MEASUREMENT_NOISE_FOOT = np.array([7e-3, 3e-3, 2e-3])
MEASUREMENT_NOISE_SCALE = 2.0

# Covariance ellipse visualization
XY_COVARIANCE_ELLIPSE_SIGMA = 2.0
XY_COVARIANCE_ELLIPSE_STRIDE = 29

### 2. Synthetic Data Generation

We start with synthetic data, where the ground-truth robot state is known. The robot follows a circular trajectory using a trotting gait with alternating diagonal contacts and short flight phases (no foot in contact with the ground), generating simulated IMU measurements and foot contact information.

This data will be used throughout the tutorial before moving to real-world data. Feel free to check the code, but you **do not need to modify** anything here.

> ⚠️ **Note**: Regenerate the dataset using `generate_circular_trajectory_csv(...)` every time you change any of these settings:
>
> - `USE_IMU_NOISE`, `ACC_NOISE_STD`, `GYRO_NOISE_STD`
> - `USE_IMU_BIAS`, `ACC_BIAS`, `GYRO_BIAS`
> - `FOOT_SLIP_STD`, `FOOT_SLIP_ENABLE_Z`
>
> These settings affect the generated synthetic dataset, which is exported as a `.csv` file and then used by the estimator.

In [5]:
# =============================================================================
# 2. Synthetic data generation
# =============================================================================

LEG_NAMES = ["FL", "FR", "BL", "BR"]
FOOT_OFFSETS_BODY = {
    "FL": np.array([0.2, 0.2, -0.3]),
    "FR": np.array([0.2, -0.2, -0.3]),
    "BL": np.array([-0.2, 0.2, -0.3]),
    "BR": np.array([-0.2, -0.2, -0.3]),
}

def quaternion_wxyz_to_rotation_matrix(quaternion_wxyz: np.ndarray) -> np.ndarray:
    """Convert [w, x, y, z] quaternion to a rotation matrix."""
    return R.from_quat([
        quaternion_wxyz[1],
        quaternion_wxyz[2],
        quaternion_wxyz[3],
        quaternion_wxyz[0],
    ]).as_matrix()
    
def simulate_imu(a_w, q_wxyz, dt_s, yaw_rate=None,
                 accel_noise_std=ACC_NOISE_STD, gyro_noise_std=GYRO_NOISE_STD):
    """Simulate IMU: transform accel to body frame + optional noise/bias. Frequency of 100 Hz."""

    acc = np.zeros_like(a_w)
    for i in range(len(q_wxyz)):
        R = quaternion_wxyz_to_rotation_matrix(q_wxyz[i])
        acc[i] = R.T @ (a_w[i] - GRAVITY_WORLD)  # world → body, remove gravity

    # Gyro: only yaw rate for this trajectory
    gyro = (np.zeros((len(q_wxyz), 3)) if yaw_rate is None
            else np.column_stack([np.zeros(len(q_wxyz)), np.zeros(len(q_wxyz)), yaw_rate]))

    # Add noise and bias if enabled
    if USE_IMU_NOISE:
        acc += np.random.normal(0.0, accel_noise_std, acc.shape)
        gyro += np.random.normal(0.0, gyro_noise_std, gyro.shape)
    if USE_IMU_BIAS:
        acc += ACC_BIAS
        gyro += GYRO_BIAS

    return acc, gyro

def create_alternating_trot_contact_schedule(
    num_samples: int,
    contact_duration_samples: int,
    no_contact_duration_samples: int = 5,
) -> list[dict[str, bool]]:
    """Trot pattern: FL+BR → flight → FR+BL → flight. Contact switches at ~4 Hz (duration=25)."""
    
    cycle = 2 * (contact_duration_samples + no_contact_duration_samples)
    schedule = []
    for i in range(num_samples):
        p = i % cycle

        if p < contact_duration_samples:
            contacts = {"FL": True, "BR": True, "FR": False, "BL": False}
        elif p < contact_duration_samples + no_contact_duration_samples:
            contacts = {"FL": False, "BR": False, "FR": False, "BL": False}
        elif p < 2 * contact_duration_samples + no_contact_duration_samples:
            contacts = {"FL": False, "BR": False, "FR": True, "BL": True}
        else:
            contacts = {"FL": False, "BR": False, "FR": False, "BL": False}

        schedule.append(contacts)

    return schedule

def build_feet_and_contacts(t, p_w, q_wxyz, contact_duration_samples, swing_foot_height_m):
    """Foot positions (body frame) + contact flags."""

    sched = create_alternating_trot_contact_schedule(len(t), contact_duration_samples, 5)
    foot_world, foot_pos, contact = {l: None for l in LEG_NAMES}, {}, {}

    for leg, offset_b in FOOT_OFFSETS_BODY.items():
        foot_b, c = np.zeros((len(t), 3)), np.zeros(len(t), int)

        for i in range(len(t)):
            R = quaternion_wxyz_to_rotation_matrix(q_wxyz[i])
            in_contact = sched[i][leg]; c[i] = int(in_contact)

            if in_contact:
                # Fix foot in world at contact start + optional slip
                if foot_world[leg] is None or not sched[i - 1][leg]:
                    foot_world[leg] = p_w[i] + R @ offset_b
                if FOOT_SLIP_STD > 0.0:
                    z = np.random.normal(0.0, FOOT_SLIP_STD * 0.1) if FOOT_SLIP_ENABLE_Z else 0.0
                    foot_world[leg] += np.array([
                        np.random.normal(0.0, FOOT_SLIP_STD),
                        np.random.normal(0.0, FOOT_SLIP_STD),
                        z,
                    ])
                foot_b[i] = R.T @ (foot_world[leg] - p_w[i])  # world → body

            else:
                # Simple swing motion
                phase = (i % contact_duration_samples) / contact_duration_samples
                foot_b[i] = offset_b + np.array([-0.2 + 0.4*phase, 0.0, swing_foot_height_m*np.sin(np.pi*phase)])
                foot_world[leg] = None

        foot_pos[leg], contact[leg] = foot_b, c

    return foot_pos, contact

def build_dataframe(
    t: np.ndarray,
    acc: np.ndarray,
    gyro: np.ndarray,
    p_w: np.ndarray,
    q_wxyz: np.ndarray,
    foot_pos: dict[str, np.ndarray],
    contact: dict[str, np.ndarray],
) -> pd.DataFrame:
    data = {
        "ts": t,
        "acc_x": acc[:, 0], "acc_y": acc[:, 1], "acc_z": acc[:, 2],
        "gyro_x": gyro[:, 0], "gyro_y": gyro[:, 1], "gyro_z": gyro[:, 2],
        "gt_pos_x": p_w[:, 0], "gt_pos_y": p_w[:, 1], "gt_pos_z": p_w[:, 2],
        "gt_rot_w": q_wxyz[:, 0], "gt_rot_x": q_wxyz[:, 1], "gt_rot_y": q_wxyz[:, 2], "gt_rot_z": q_wxyz[:, 3],
    }
    for leg in LEG_NAMES:
        for k, ax in enumerate(["x", "y", "z"]):
            data[f"{leg}_{ax}"] = foot_pos[leg][:, k]
        data[f"{leg}_contact"] = contact[leg]
    return pd.DataFrame(data)

def generate_circular_motion(total_time_s, dt_s, freq_hz=0.2):
    """Simple circular trajectory with small vertical motion."""

    t = np.arange(0.0, total_time_s, dt_s)
    omega, omega_z = 2 * np.pi * freq_hz, 4 * np.pi * freq_hz

    p_w = np.column_stack([np.cos(omega*t), np.sin(omega*t), 0.1*np.sin(omega_z*t) + 0.3])
    v_w = np.column_stack([-omega*np.sin(omega*t), omega*np.cos(omega*t), 0.1*omega_z*np.cos(omega_z*t)])
    a_w = np.column_stack([-omega**2*np.cos(omega*t), -omega**2*np.sin(omega*t), -0.1*omega_z**2*np.sin(omega_z*t)])

    yaw = np.arctan2(v_w[:, 1], v_w[:, 0])
    quat = R.from_euler("zyx", np.column_stack([yaw, np.zeros(len(t)), np.zeros(len(t))])).as_quat()
    q_wxyz = np.column_stack([quat[:, 3], quat[:, 0], quat[:, 1], quat[:, 2]])

    return t, p_w, q_wxyz, np.full(len(t), omega), a_w

def generate_circular_trajectory_csv(
    total_time_s: float = 10.0,
    dt_s: float = 0.01,
    csv_path: str = "datasets/quadruped_sim.csv",
    overwrite_existing_file: bool = True,
    contact_duration_samples: int = 25,
    swing_foot_height_m: float = 0.1,
    accel_noise_std: float = ACC_NOISE_STD,
    gyro_noise_std: np.ndarray = GYRO_NOISE_STD,
) -> str:
    """Generate synthetic trajectory + IMU + foot data and save to CSV."""

    if os.path.exists(csv_path) and not overwrite_existing_file:
        print(f"Reusing existing data: {csv_path}")
        return csv_path

    # Generate motion + IMU + foot/contact data
    t, p_w, q_wxyz, yaw_rate, a_w = generate_circular_motion(total_time_s, dt_s)
    acc, gyro = simulate_imu(a_w, q_wxyz, dt_s, yaw_rate, accel_noise_std, gyro_noise_std)
    foot_pos, contact = build_feet_and_contacts(t, p_w, q_wxyz, contact_duration_samples, swing_foot_height_m)

    # Save to CSV
    df = build_dataframe(t, acc, gyro, p_w, q_wxyz, foot_pos, contact)
    df.to_csv(csv_path, index=False)

    print(f"Simulation data saved to {csv_path}")
    return csv_path

### 3. Plotting Utilities

A collection of plotting functions is provided in this notebook. They are used in the final section to visualize and evaluate your results.

Feel free to check the code, but you **do not need to modify** anything here.

In [6]:
# =============================================================================
# 3. Plotting
# =============================================================================
    
def plot_xyz(ax, t: np.ndarray, data: np.ndarray, title: str, ylabel: str, labels: list[str] | None = None) -> None:
    if labels is None:
        labels = ["X", "Y", "Z"]
    for i, label in enumerate(labels):
        ax.plot(t, data[:, i], label=label)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True)

def plot_imu(t: np.ndarray, acc: np.ndarray, gyro: np.ndarray) -> None:
    fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    plot_xyz(axes[0], t, acc, "Accelerometer (Body Frame)", "Linear acceleration (m/s²)")
    plot_xyz(axes[1], t, gyro, "Gyroscope (Body Frame)", "Angular velocity (rad/s)")
    axes[1].set_xlabel("Time (s)")
    plt.tight_layout(rect=[0, 0.12, 1, 1])
    plt.show()

def plot_contact_states(
    time_s: np.ndarray,
    contact_by_leg: dict[str, np.ndarray],
    leg_names: list[str],
) -> None:
    """Show which feet are in stance (1) or swing (0)."""
    plt.figure(figsize=(9, 4))
    ordered_legs = ["FL", "FR", "BL", "BR"]

    for leg_index, leg_name in enumerate(ordered_legs):
        vertical_offset = 1.2 * (len(ordered_legs) - 1 - leg_index)
        plt.step(
            time_s,
            contact_by_leg[leg_name].astype(float) + vertical_offset,
            where="post",
            label=leg_name,
        )

    plt.title("Foot Contact States")
    plt.xlabel("Time (s)")
    plt.ylabel("Contact (0=swing, 1=stance)")
  
    yticks_pos = [1.2 * (len(ordered_legs) - 1 - i) + 0.5 for i in range(len(ordered_legs))]
    plt.yticks(yticks_pos, ordered_legs)

    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=4, framealpha=0.9)
    plt.grid(True)
    plt.tight_layout(rect=[0, 0.12, 1, 1])
    plt.show()

def plot_trajectory_2d(gt: np.ndarray, est: np.ndarray, y_idx: int, title: str, ylabel: str, baseline: np.ndarray | None = None) -> None:
    series = [(gt, "Ground Truth", "tab:green", "-"), (est, "EKF (This Work)", "tab:red", "--")]
    if baseline is not None:
        series.insert(1, (baseline, "InEKF (Robot)", "tab:blue", "-"))
    all_points = [gt, est] + ([baseline] if baseline is not None else [])
    all_points = np.vstack(all_points)
    x_range = max(all_points[:, 0].max() - all_points[:, 0].min(), 1e-6)
    y_range = max(all_points[:, y_idx].max() - all_points[:, y_idx].min(), 1e-6)
    flat = y_range / x_range < 0.45
    fig, ax = plt.subplots(figsize=(8, 3.4) if flat else (6, 6))
    bottom = 0.60 if flat else 0.26
    legend_y = -0.62 if flat else -0.22
    xlabel_pad = 2 if flat else 16
    for traj, label, color, ls in series:
        ax.plot(traj[:, 0], traj[:, y_idx], color=color, linestyle=ls, label=label)
        ax.scatter(traj[0, 0], traj[0, y_idx], facecolors="none", edgecolors=color, marker="o", s=60)
        ax.scatter(traj[-1, 0], traj[-1, y_idx], c=color, marker="o", s=60)
    marker_legend = [
        Line2D([0], [0], marker="o", color="black", linestyle="None", markerfacecolor="none", label="Start"),
        Line2D([0], [0], marker="o", color="black", linestyle="None", markerfacecolor="black", label="End"),
    ]
    handles, labels = ax.get_legend_handles_labels()
    ax.set_title(title)
    ax.set_xlabel("X (m)", labelpad=xlabel_pad)
    ax.set_ylabel(ylabel)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True)
    ax.legend(handles + marker_legend, labels + ["Start", "End"], loc="upper center", bbox_to_anchor=(0.5, legend_y), ncol=2, framealpha=0.9)
    fig.subplots_adjust(bottom=bottom)
    plt.show()

def plot_xy_trajectory_with_covariance_ellipses(
    gt, est, covariance_history, baseline=None,
    ellipse_stride=XY_COVARIANCE_ELLIPSE_STRIDE,
    ellipse_scale=XY_COVARIANCE_ELLIPSE_SIGMA,
    max_samples=None,
) -> None:
    """Plot XY trajectory with covariance ellipses."""

    from matplotlib.patches import Ellipse

    n = min(len(gt), len(est), len(covariance_history))
    if max_samples is not None:
        n = min(n, max_samples)

    gt, est = gt[:n], est[:n]
    covariance_history = covariance_history[:n]
    if baseline is not None:
        baseline = baseline[:n]

    series = [(gt, "Ground Truth", "tab:green", "-"), (est, "EKF", "tab:red", "--")]
    if baseline is not None:
        series.insert(1, (baseline, "Baseline", "tab:blue", "-"))

    fig, ax = plt.subplots(figsize=(6, 6))

    for traj, label, color, style in series:
        ax.plot(traj[:, 0], traj[:, 1], color=color, linestyle=style, label=label)
        ax.scatter(traj[0, 0], traj[0, 1], facecolors="none", edgecolors=color, s=60)
        ax.scatter(traj[-1, 0], traj[-1, 1], color=color, s=60)

    for i in range(0, n, ellipse_stride):
        P_xy = covariance_history[i][6:8, 6:8]
        vals, vecs = np.linalg.eigh(P_xy)
        vals = np.maximum(vals, 1e-12)
        order = np.argsort(vals)[::-1]
        vals, vecs = vals[order], vecs[:, order]

        angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
        width = 2 * ellipse_scale * np.sqrt(vals[0])
        height = 2 * ellipse_scale * np.sqrt(vals[1])

        ax.add_patch(Ellipse(
            est[i, :2], width, height, angle=angle,
            facecolor="tab:red", edgecolor="tab:red",
            alpha=0.18, linewidth=1.0,
        ))

    ax.set_title("Top View (XY) with Covariance Ellipses")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True)
    ax.legend()
    plt.show()

def compute_orientation_error(
    time_s: np.ndarray,
    estimated_base_states: np.ndarray,
    ground_truth_quaternion_wxyz: np.ndarray,
) -> np.ndarray:
    """Return orientation error as axis-angle vectors."""
    estimated_rotations = [
        R.from_matrix(estimated_base_states[i, :9].reshape(3, 3))
        for i in range(len(time_s))
    ]
    true_rotations = [
        R.from_quat([q[1], q[2], q[3], q[0]])
        for q in ground_truth_quaternion_wxyz
    ]
    rotation_error_vectors = np.array(
        [(estimated.inv() * true).as_rotvec() for estimated, true in zip(estimated_rotations, true_rotations)]
    )

    return rotation_error_vectors

def print_summary(
    ground_truth_position: np.ndarray,
    time_s: np.ndarray,
    position_error: np.ndarray,
    rotation_error_vectors: np.ndarray,
) -> None:
    """Print a few summary metrics for quick comparison."""
    total_distance_m = np.sum(np.linalg.norm(np.diff(ground_truth_position, axis=0), axis=1))
    average_speed_mps = total_distance_m / (time_s[-1] - time_s[0])
    position_rmse_m = np.sqrt(np.mean(position_error**2))
    position_rmse_xyz_m = np.sqrt(np.mean(position_error**2, axis=0))
    orientation_rmse_rad = np.sqrt(np.mean(np.linalg.norm(rotation_error_vectors, axis=1) ** 2))

    print("Performance Summary:")
    print(f" RMSE Position    : {position_rmse_m:.3f} m")
    print(
        f" RMSE X/Y/Z       : {position_rmse_xyz_m[0]:.3f}, "
        f"{position_rmse_xyz_m[1]:.3f}, {position_rmse_xyz_m[2]:.3f} m"
    )
    print(f" RMSE Orientation : {orientation_rmse_rad:.3f} rad")
    print(f" Total Distance   : {total_distance_m:.2f} m")
    print(f" Average Velocity : {average_speed_mps:.2f} m/s")

### 4. Data Loading

In this section, we load the dataset used by the estimator. There are two options: loading either synthetic or real data from a `.csv` file containing IMU measurements, ground truth (if available), and foot/contact information.

Feel free to check the code, but you **do not need to modify** anything here.

In [7]:
# =============================================================================
# 4. Data loading
# =============================================================================

def load_simulated_dataset(csv_path: str = SIMULATED_DATA_CSV_PATH) -> dict:
    csv_path = generate_circular_trajectory_csv(csv_path=csv_path)
    df = pd.read_csv(csv_path)
    return {
        "time_s": df["ts"].values,
        "accelerometer": df[["acc_x", "acc_y", "acc_z"]].values,
        "gyroscope": df[["gyro_x", "gyro_y", "gyro_z"]].values,
        "foot_position_body_by_leg": {
            leg: df[[f"{leg}_x", f"{leg}_y", f"{leg}_z"]].values for leg in LEG_NAMES
        },
        "contact_by_leg": {
            leg: df[f"{leg}_contact"].astype(bool).values for leg in LEG_NAMES
        },
        "ground_truth_position": df[["gt_pos_x", "gt_pos_y", "gt_pos_z"]].values,
        "ground_truth_quaternion_wxyz": df[["gt_rot_w", "gt_rot_x", "gt_rot_y", "gt_rot_z"]].values,
        "baseline_position": None,
        "dt_s": np.diff(df["ts"].values, prepend=df["ts"].values[0]),
    }

def load_real_dataset(csv_path: str) -> dict:
    df = pd.read_csv(csv_path)
    df["timestamp"] = pd.to_datetime(df["ts"], unit="s", utc=True)
    t0 = df["timestamp"].iloc[0]
    df["t"] = (df["timestamp"] - t0).dt.total_seconds()
    df[["gt_pos_x", "gt_pos_y", "gt_pos_z"]] /= 1000.0
    return {
        "time_s": df["t"].values,
        "accelerometer": df[["acc_x", "acc_y", "acc_z"]].values,
        "gyroscope": df[["gyro_x", "gyro_y", "gyro_z"]].values,
        "foot_position_body_by_leg": {
            leg: df[[f"{leg}_x", f"{leg}_y", f"{leg}_z"]].values for leg in LEG_NAMES
        },
        "contact_by_leg": {
            leg: df[f"{leg}_contact"].astype(bool).values for leg in LEG_NAMES
        },
        "ground_truth_position": df[["gt_pos_x", "gt_pos_y", "gt_pos_z"]].values,
        "ground_truth_quaternion_wxyz": df[["gt_rot_w", "gt_rot_x", "gt_rot_y", "gt_rot_z"]].values,
        "baseline_position": df[["state_est_pos_x", "state_est_pos_y", "state_est_pos_z"]].values,
        "baseline_quaternion_wxyz": df[["state_est_rot_w", "state_est_rot_x", "state_est_rot_y", "state_est_rot_z"]].values,
        "dt_s": np.diff(df["t"].values, prepend=df["t"].values[0]),
    }

### 5. 🔧 EKF Implementation 🔧 

In this section, you will implement the core EKF steps.

**To Do**: Complete the **predict** and **update** methods using the provided structure and comments as guidance. These are the key components of the estimator.

In [11]:
# =============================================================================
# 5. Extended Kalman Filter
# State = [R(9), v(3), p(3), active stance feet in world frame]
# =============================================================================

class QuadrupedEKF:
    """EKF with extra states for feet that are currently in stance."""

    def __init__(self) -> None:
        # Nominal state: [R(9), v(3), p(3)]
        self.nominal_base_state_size = 15
        self.state_vector = np.zeros(self.nominal_base_state_size)
        self.state_vector[:9] = np.eye(3).reshape(-1)

        # Error-state covariance: [δθ(3), δv(3), δp(3)]
        self.base_state_size = 9
        self.full_covariance = np.eye(self.base_state_size) * 1e-3

        self.process_noise_floating_base = np.diag([
            PROCESS_NOISE_ROT, PROCESS_NOISE_ROT, PROCESS_NOISE_ROT,
            PROCESS_NOISE_VEL[0], PROCESS_NOISE_VEL[1], PROCESS_NOISE_VEL[2],
            PROCESS_NOISE_POS[0], PROCESS_NOISE_POS[1], PROCESS_NOISE_POS[2],
        ])

        self.measurement_noise_foot = MEASUREMENT_NOISE_SCALE * np.diag(MEASUREMENT_NOISE_FOOT)
        self.gravity_world = GRAVITY_WORLD.copy()

        # Map each stance foot to its location inside the nominal state vector.
        self.foot_state_start_index: dict[str, int] = {}
        
    @staticmethod
    def skew(vector_3d: np.ndarray) -> np.ndarray:
        """Return the skew matrix used for cross products."""
        return np.array(
            [
                [0.0, -vector_3d[2], vector_3d[1]],
                [vector_3d[2], 0.0, -vector_3d[0]],
                [-vector_3d[1], vector_3d[0], 0.0],
            ]
        )
    def project_rotation_to_SO3(self) -> None:
        R_mat = self.state_vector[:9].reshape(3, 3)
        U, _, Vt = np.linalg.svd(R_mat)
        R_proj = U @ Vt
        if np.linalg.det(R_proj) < 0:
            U[:, -1] *= -1
            R_proj = U @ Vt
        self.state_vector[:9] = R_proj.reshape(-1)

    def add_foot_state(self, leg_name: str, measured_foot_position_body: np.ndarray) -> None:
        """Add a stance foot as a fixed world-frame anchor."""
    
        if leg_name in self.foot_state_start_index:
            return
    
        R_wb = self.state_vector[:9].reshape(3, 3)
        p_w = self.state_vector[12:15]
        foot_position_world = p_w + R_wb @ measured_foot_position_body
    
        # Add foot to nominal state [R(9), v(3), p(3), feet...]
        foot_state_index = len(self.state_vector)
        self.foot_state_start_index[leg_name] = foot_state_index
        self.state_vector = np.hstack([self.state_vector, foot_position_world])
    
        # Add corresponding 3D foot error covariance after [δθ, δv, δp]
        n_old = len(self.full_covariance)
        expanded_covariance = np.zeros((n_old + 3, n_old + 3))
        expanded_covariance[:n_old, :n_old] = self.full_covariance
        expanded_covariance[-3:, -3:] = np.eye(3) * 1e-2
        self.full_covariance = expanded_covariance
    
    
    def remove_foot_state(self, leg_name: str) -> None:
        """Remove foot state and its covariance when contact ends."""
    
        if leg_name not in self.foot_state_start_index:
            return
    
        removed_state_index = self.foot_state_start_index.pop(leg_name)
    
        # Remove from nominal state
        keep_state = np.ones(len(self.state_vector), dtype=bool)
        keep_state[removed_state_index:removed_state_index + 3] = False
        self.state_vector = self.state_vector[keep_state]
    
        # Remove matching foot block from covariance
        removed_cov_index = 9 + (removed_state_index - 15)
        keep_cov = np.ones(len(self.full_covariance), dtype=bool)
        keep_cov[removed_cov_index:removed_cov_index + 3] = False
        self.full_covariance = self.full_covariance[keep_cov][:, keep_cov]
    
        # Shift stored nominal indices after removal
        for other_leg, other_index in list(self.foot_state_start_index.items()):
            if other_index > removed_state_index:
                self.foot_state_start_index[other_leg] = other_index - 3

    def predict(self, accelerometer: np.ndarray, gyroscope: np.ndarray, dt_s: float) -> None:
        # ------------------------------------------------------------------
        # TASK: Implement the main EKF prediction equations.
        #
        # Fill in the key steps marked with TODO.
        #
        # State convention:
        #   nominal state = [R(9), v(3), p(3), foot states...]
        #   error state   = [δθ(3), δv(3), δp(3), foot errors...]
        #
        # Important:
        #   - Gyroscope propagates orientation.
        #   - Accelerometer measures specific force, so gravity is added back.
        #   - Covariance uses 3D rotation error δθ, not 9 rotation entries.
        # ------------------------------------------------------------------
    
        R_wb = self.state_vector[:9].reshape(3, 3)
        v_w = self.state_vector[9:12]
        p_w = self.state_vector[12:15]
    
        # TODO 1: Propagate orientation using the gyroscope
        ... 
    
        # TODO 2: Rotate acceleration to world frame and add gravity
        ...
    
        # TODO 3: Integrate velocity and position
        ...
    
        self.state_vector[:9] = R_pred.reshape(-1)
        self.state_vector[9:12] = v_pred
        self.state_vector[12:15] = p_pred
        self.project_rotation_to_SO3()
    
        # TODO 4: Build the main error-state Jacobian blocks
        n = len(self.full_covariance)
        F = np.eye(n)
    
        F[0:3, 0:3] = ...
        F[3:6, 0:3] = ...
        F[6:9, 0:3] = ...
        F[6:9, 3:6] = ...
    
        Q = np.zeros((n, n))
        Q[:9, :9] = self.process_noise_floating_base
    
        # TODO 5: Propagate covariance
        self.full_covariance = ...
        self.full_covariance = 0.5 * (self.full_covariance + self.full_covariance.T)

    def update(self, leg_name: str, measured_foot_position_body: np.ndarray) -> None:
        # ------------------------------------------------------------------
        # TASK: Implement the main EKF correction equations.
        #
        # Fill in the key steps marked with TODO.
        #
        # Measurement model:
        #   p_foot_pred = p_base + R_wb @ p_foot_body
        #   r = p_foot_pred - p_foot_stored
        #
        # State convention:
        #   nominal state = [R(9), v(3), p(3), foot states...]
        #   error state   = [δθ(3), δv(3), δp(3), foot errors...]
        #
        # Important:
        #   - Rotation correction uses exp([δθ]x).
        #   - Position error is stored at covariance indices 6:9.
        #   - Foot orientation correction is used only for clean simulated data.
        # ------------------------------------------------------------------
    
        if leg_name not in self.foot_state_start_index:
            return
    
        foot_state_index = self.foot_state_start_index[leg_name]
        foot_cov_index = 9 + (foot_state_index - 15)
    
        R_wb = self.state_vector[:9].reshape(3, 3)
        p_w = self.state_vector[12:15]
        foot_w = self.state_vector[foot_state_index:foot_state_index + 3]
    
        # TODO 1: Compute predicted foot position and residual
        predicted_foot_world = ...
        residual = ...
    
        # TODO 2: Build measurement Jacobian H
        n = len(self.full_covariance)
        H = np.zeros((3, n))
    
        if not USE_REAL_DATA:
            H[:, 0:3] = ...                             # orientation
        H[:, 6:9] = ...                                 # position
        H[:, foot_cov_index:foot_cov_index + 3] = ...   # foot state
    
        # TODO 3: Compute Kalman gain
        S = ...
        K = ...
    
        # TODO 4: Compute and inject error-state correction
        delta_x = ...
        delta_theta = delta_x[0:3]
    
        self.state_vector[:9] = ...     # R
        self.state_vector[9:12] += ...  # v
        self.state_vector[12:15] += ... # p
    
        for leg, state_idx in self.foot_state_start_index.items():
            cov_idx = 9 + (state_idx - 15)
            self.state_vector[state_idx:state_idx + 3] += delta_x[cov_idx:cov_idx + 3]
    
        self.project_rotation_to_SO3()
    
        # TODO 5: Update covariance (Joseph form)
        I = np.eye(n)
        self.full_covariance = ...
        self.full_covariance = 0.5 * (self.full_covariance + self.full_covariance.T)

### 6. Helper Functions

The following helper functions wrap the existing loading, EKF execution, evaluation, and plotting code into smaller reusable blocks.

Feel free to check the code, but you **do not need to modify** anything here.

In [9]:
# =============================================================================
# 6. Helper Functions
# =============================================================================

def load_selected_dataset(use_real_data: bool = USE_REAL_DATA) -> dict:
    """Load simulation first by default; real data is used only in the last task."""
    return load_real_dataset(REAL_DATA_CSV_PATH) if use_real_data else load_simulated_dataset(SIMULATED_DATA_CSV_PATH)


def initialize_filter_from_ground_truth_old(dataset: dict) -> QuadrupedEKF:
    """Initialize the EKF close to the true initial pose."""
    ekf = QuadrupedEKF()
    time_s = dataset["time_s"]
    ground_truth_position = dataset["ground_truth_position"]
    ground_truth_quaternion_wxyz = dataset["ground_truth_quaternion_wxyz"]

    initial_rotation_world_from_body = quaternion_wxyz_to_rotation_matrix(
        ground_truth_quaternion_wxyz[0]
    )
    ekf.state_vector[:9] = initial_rotation_world_from_body.reshape(-1)
    ekf.state_vector[12:15] = ground_truth_position[0]
    initial_velocity_world = np.gradient(ground_truth_position, time_s, axis=0)[0]
    ekf.state_vector[9:12] = initial_velocity_world
    return ekf

def initialize_filter_from_ground_truth(dataset: dict) -> QuadrupedEKF:
    """Initialize nominal state from the first ground-truth pose."""

    ekf = QuadrupedEKF()

    time_s = dataset["time_s"]
    p_gt = dataset["ground_truth_position"]
    q_gt = dataset["ground_truth_quaternion_wxyz"]

    # Nominal state: [R(9), v(3), p(3), feet...]
    ekf.state_vector = np.zeros(15)
    ekf.state_vector[:9] = quaternion_wxyz_to_rotation_matrix(q_gt[0]).reshape(-1)
    ekf.state_vector[9:12] = np.gradient(p_gt, time_s, axis=0)[0]
    ekf.state_vector[12:15] = p_gt[0]

    # Error-state covariance: [δθ(3), δv(3), δp(3), feet...]
    ekf.full_covariance = np.eye(9) * 1e-6

    return ekf

def run_ekf_on_dataset(
    dataset: dict,
    disable_foot_updates: bool = False,
) -> dict:
    """Run the EKF over a dataset and return signals used for analysis/plots."""

    # Unpack the dataset signals used by the filter.
    time_s = dataset["time_s"]
    accelerometer = dataset["accelerometer"]
    gyroscope = dataset["gyroscope"]
    foot_position_body_by_leg = dataset["foot_position_body_by_leg"]
    contact_by_leg = dataset["contact_by_leg"]
    ground_truth_position = dataset["ground_truth_position"]
    ground_truth_quaternion_wxyz = dataset["ground_truth_quaternion_wxyz"]
    dt_s = dataset["dt_s"]

    # Initialize the filter from the known initial ground-truth state.
    ekf = initialize_filter_from_ground_truth(dataset)

    estimated_state_history = []
    covariance_history = []
    active_foot_count_history = []

    for sample_index in range(len(time_s)):
        # Prediction step: propagate the base state using IMU measurements.
        ekf.predict(
            accelerometer[sample_index],
            gyroscope[sample_index],
            dt_s[sample_index],
        )

        # Add/remove foot states depending on the current contact state.
        for leg_name in LEG_NAMES:
            is_in_contact = contact_by_leg[leg_name][sample_index]
            measured_foot_position_body = foot_position_body_by_leg[leg_name][sample_index]

            if is_in_contact and leg_name not in ekf.foot_state_start_index:
                ekf.add_foot_state(leg_name, measured_foot_position_body)
            elif (not is_in_contact) and leg_name in ekf.foot_state_start_index:
                ekf.remove_foot_state(leg_name)

        # Correction step: use active stance feet as measurements.
        if not disable_foot_updates:
            for leg_name in list(ekf.foot_state_start_index.keys()):
                measured_foot_position_body = foot_position_body_by_leg[leg_name][sample_index]
                ekf.update(leg_name, measured_foot_position_body)

        # Store filter state for later plotting and evaluation.
        estimated_state_history.append(ekf.state_vector.copy())
        covariance_history.append(ekf.full_covariance.copy())
        active_foot_count_history.append(len(ekf.foot_state_start_index))

    # Extract base position and compute estimation errors.
    estimated_base_state_history = np.array([state[:15] for state in estimated_state_history])
    estimated_position = estimated_base_state_history[:, 12:15]
    position_error = estimated_position - ground_truth_position

    rotation_error_vectors = compute_orientation_error(
        time_s,
        estimated_base_state_history,
        ground_truth_quaternion_wxyz,
    )

    return {
        "time_s": time_s,
        "estimated_state_history": estimated_state_history,
        "estimated_base_state_history": estimated_base_state_history,
        "estimated_position": estimated_position,
        "covariance_history": covariance_history,
        "position_error": position_error,
        "rotation_error_vectors": rotation_error_vectors,
        "active_foot_count_history": np.array(active_foot_count_history),
    }


def align_baseline_if_available(dataset: dict, use_real_data: bool = USE_REAL_DATA) -> np.ndarray | None:
    """INEKF Baseline alignment."""
    baseline_position = dataset["baseline_position"]
    if use_real_data and baseline_position is not None:
        ground_truth_position = dataset["ground_truth_position"]
        ground_truth_quaternion_wxyz = dataset["ground_truth_quaternion_wxyz"]
        R_gt0 = quaternion_wxyz_to_rotation_matrix(ground_truth_quaternion_wxyz[0])
        R_base0 = quaternion_wxyz_to_rotation_matrix(dataset["baseline_quaternion_wxyz"][0])
        R_align = R_gt0 @ R_base0.T
        t_align = ground_truth_position[0] - R_align @ baseline_position[0]
        return (R_align @ baseline_position.T).T + t_align
    return baseline_position


def compute_rmse(error: np.ndarray) -> np.ndarray:
    """Root-mean-square error per axis."""
    return np.sqrt(np.mean(error**2, axis=0))
    

def visualize_state_matrix(ekf: QuadrupedEKF) -> None:
    """Visualize the current state as blocks: R, v, p, and active foot states."""
    R_wb = ekf.state_vector[:9].reshape(3, 3)
    v_w = ekf.state_vector[9:12].reshape(3, 1)
    p_w = ekf.state_vector[12:15].reshape(3, 1)
    print("R_world_body =")
    print(np.round(R_wb, 3))
    print("v_world =", np.round(v_w.ravel(), 3))
    print("p_world =", np.round(p_w.ravel(), 3))
    print("active foot states:")
    if len(ekf.foot_state_start_index) == 0:
        print("  none")
    for leg_name, start_idx in ekf.foot_state_start_index.items():
        print(f"  {leg_name}:", np.round(ekf.state_vector[start_idx:start_idx + 3], 3))

## Part B: Student Tasks

Run each section in order and answer the questions.

### Task 1 — Load the simulation data

**Practical task:** load the simulated dataset using `load_selected_dataset`. 

**Question:** What advantages does simulation provide when developing and debugging a state estimator?

**Answer:** TODO

In [ ]:
USE_REAL_DATA = ... # TODO
dataset = ...       # TODO

print("Loaded simulated dataset")
print("Number of samples:", len(dataset["time_s"]))
print("Duration [s]:", dataset["time_s"][-1] - dataset["time_s"][0])

plot_imu(dataset["time_s"], dataset["accelerometer"], dataset["gyroscope"])
plot_contact_states(dataset["time_s"], dataset["contact_by_leg"], LEG_NAMES)

### Task 2 — Visualize the robot state matrix

**Practical task:** inspect the EKF state at initialization.

**💡 Hint:** Initialize the EKF using `initialize_filter_from_ground_truth` and print the state using `visualize_state_matrix`.  
Notice that no foot states are active at initialization. Add the stance feet for the first time step using the contact information and visualize the state again.

**Questions:**  
1. What is the structure of the state vector?  
2. Why are foot positions only included for legs in contact?  

**Answers:**  
1. TODO
2. TODO

In [ ]:
ekf_preview = ... # TODO

# Look at the first time step
sample_index = 0

# Loop over all legs and check which ones are in contact
for leg_name in LEG_NAMES:
    # If the leg is in contact at this time step
    if dataset["contact_by_leg"][leg_name][sample_index]:
        # Add this foot as a state in the EKF
        ekf_preview.add_foot_state(
            leg_name,
            dataset["foot_position_body_by_leg"][leg_name][sample_index],
        )

# Print the state
... # TODO

### Task 3 — Implement the propagation step

**Practical task:** Implement the `predict` method of the `QuadrupedEKF` class. Follow the steps outlined in the comments inside the method.

After implementing it, run the EKF on the simulated dataset and check the result using the RMSE and top-view trajectory plot.

**Question:** Why must accelerometer data be rotated into the world frame, and why is
gravity added/subtracted?

**Answer:** TODO

In [ ]:
results_nominal = run_ekf_on_dataset(dataset, disable_foot_updates=True)

print("Nominal simulation run complete.")
print("Position RMSE [x, y, z] (m):", np.round(compute_rmse(results_nominal["position_error"]), 4))
plot_trajectory_2d(dataset["ground_truth_position"][1:], results_nominal["estimated_position"][1:], 1, "Top View (XY)", "Y (m)")
plot_trajectory_2d(dataset["ground_truth_position"][1:], results_nominal["estimated_position"][1:], 2, "Side View (XZ)", "Z (m)")

### Task 4 — Add IMU noise

**Practical task:** Enable IMU noise in the simulated dataset and vary the accelerometer and gyroscope noise values.

IMU noise is added during synthetic CSV generation, so after changing `USE_IMU_NOISE`, `ACC_NOISE_STD`, or `GYRO_NOISE_STD`, you must regenerate the dataset using `generate_circular_trajectory_csv(...)`.

In this task, foot updates are disabled (`disable_foot_updates=True`), so you mainly observe drift caused by noisy IMU integration.

**💡 Hint:** Start with the default values, then increase `ACC_NOISE_STD` and/or `GYRO_NOISE_STD` and observe how the RMSE and trajectory degrade.

**Question:** How do accelerometer and gyroscope noise affect the estimated trajectory when foot updates are disabled?

**Answer:** TODO

In [ ]:
USE_IMU_NOISE = True
ACC_NOISE_STD = 0.1                               # TODO
GYRO_NOISE_STD = np.array([0.001, 0.001, 0.005])  # TODO

generate_circular_trajectory_csv(overwrite_existing_file=True)
dataset_noisy = load_selected_dataset(False)
results_noisy = run_ekf_on_dataset(dataset_noisy, disable_foot_updates=True)

print("Noisy simulation run complete.")
print_summary(
    dataset_noisy["ground_truth_position"],
    dataset_noisy["time_s"],
    results_noisy["position_error"],
    results_noisy["rotation_error_vectors"],
)

plot_trajectory_2d(dataset_noisy["ground_truth_position"][1:], results_noisy["estimated_position"][1:], 1, "Top View (XY)", "Y (m)")
plot_trajectory_2d(dataset_noisy["ground_truth_position"][1:], results_noisy["estimated_position"][1:], 2, "Side View (XZ)", "Z (m)")

USE_IMU_NOISE = False # reset for later tasks

### Task 5 — Add IMU bias

**Practical task:** Enable IMU bias and vary the accelerometer and gyroscope bias values.

IMU bias is added during synthetic CSV generation, so after changing `USE_IMU_BIAS`, `ACC_BIAS`, or `GYRO_BIAS`, you must regenerate the dataset using `generate_circular_trajectory_csv(...)`.

In this task, foot updates are disabled (`disable_foot_updates=True`), so you mainly observe how bias leads to drift in the estimate.

**💡 Hint:** Start with the given bias values, then increase `ACC_BIAS` and/or `GYRO_BIAS` and observe how the RMSE and trajectory degrade over time.

**Questions:** 
1. Why does bias usually cause worse long-term drift than zero-mean noise?
2. How can we handle IMU bias in state estimation?

**Answers:** 
1. TODO
2. TODO

In [ ]:
USE_IMU_BIAS = True
ACC_BIAS = np.array([0.08, -0.05, 0.03]) # TODO
GYRO_BIAS = np.array([0.0, 0.0, 0.02])   # TODO

generate_circular_trajectory_csv(overwrite_existing_file=True)
dataset_bias = load_selected_dataset(use_real_data=False)
results_bias = run_ekf_on_dataset(dataset_bias, disable_foot_updates=True)

print("Biased IMU simulation run complete.")
print("Position RMSE [x, y, z] (m):", np.round(compute_rmse(results_bias["position_error"]), 4))
plot_trajectory_2d(dataset_bias["ground_truth_position"][1:], results_bias["estimated_position"][1:], 1, "Top View (XY)", "Y (m)")
plot_trajectory_2d(dataset_bias["ground_truth_position"][1:], results_bias["estimated_position"][1:], 2, "Side View (XZ)", "Z (m)")

USE_IMU_BIAS = False  # reset for later tasks

### Task 6 — Implement the correction step

**Practical task:** Implement the `update` method of the `QuadrupedEKF` class. Follow the steps outlined in the comments inside the method.

After implementing it, run the EKF on the simulated dataset and check the result using the RMSE and trajectory plots.

**Questions:**
1. Why can a foot in contact be treated as a fixed point?
2. When is the system not fully observable?

**Answers:**

1. TODO
2. TODO

In [ ]:
dataset = load_selected_dataset()
results_with_correction = run_ekf_on_dataset(dataset, disable_foot_updates=False)

print("Correction-enabled simulation run complete.")
print("Position RMSE [x, y, z] (m):", np.round(compute_rmse(results_with_correction["position_error"]), 4))
plot_trajectory_2d(dataset["ground_truth_position"][1:], results_with_correction["estimated_position"][1:], 1, "Top View (XY)", "Y (m)")
plot_trajectory_2d(dataset["ground_truth_position"][1:], results_with_correction["estimated_position"][1:], 2, "Side View (XZ)", "Z (m)")

### Task 7 — Add foot slip

**Practical task:** Enable foot slip and increase `FOOT_SLIP_STD`. Regenerate the simulated dataset and compare the result to the no-slip case.

Foot slip is introduced during synthetic data generation, so after changing `FOOT_SLIP_STD` or `FOOT_SLIP_ENABLE_Z`, you must call `generate_circular_trajectory_csv(...)` again.

After modifying the parameters, run the EKF and compare the RMSE and trajectory plots.

**💡 Hint:** Start with small slip values, then increase `FOOT_SLIP_STD` to observe how the estimator degrades. Try enabling slip in the vertical direction with `FOOT_SLIP_ENABLE_Z`.

**Question:** What happens if the foot slips?


**Answer:** TODO

In [ ]:
FOOT_SLIP_STD = 0.01        # TODO
FOOT_SLIP_ENABLE_Z = False  # TODO

generate_circular_trajectory_csv(overwrite_existing_file=True)
dataset_slip = load_selected_dataset()
results_slip = run_ekf_on_dataset(dataset_slip, disable_foot_updates=False)

print("Foot-slip simulation run complete.")
print("Position RMSE [x, y, z] (m):", np.round(compute_rmse(results_slip["position_error"]), 4))
plot_trajectory_2d(dataset_slip["ground_truth_position"][1:], results_slip["estimated_position"][1:], 1, "Top View (XY)", "Y (m)")
plot_trajectory_2d(dataset_slip["ground_truth_position"][1:], results_slip["estimated_position"][1:], 2, "Side View (XZ)", "Z (m)")

#### RMSE Comparison

This table compares the **position RMSE (in meters)** for different scenarios:

- **imu_nominal**: ideal case  
- **imu_noise**: with IMU noise  
- **imu_bias**: with IMU bias  
- **imu_and_leg_odom**: with foot contact corrections  
- **foot_slip**: with slipping contacts  

Columns (`x`, `y`, `z`) show error along each axis. Lower RMSE means better estimation accuracy.

In [ ]:
# Compare RMSE across nominal, IMU noise, IMU bias, and foot slip cases
metric_table = pd.DataFrame(
    {
        "Case": ["imu_nominal", "imu_noise", "imu_bias", "imu_and_leg_odom", "foot_slip"],
        "rmse_x_m": [
            compute_rmse(results_nominal["position_error"])[0],
            compute_rmse(results_noisy["position_error"])[0],
            compute_rmse(results_bias["position_error"])[0],
            compute_rmse(results_with_correction["position_error"])[0],
            compute_rmse(results_slip["position_error"])[0],
        ],
        "rmse_y_m": [
            compute_rmse(results_nominal["position_error"])[1],
            compute_rmse(results_noisy["position_error"])[1],
            compute_rmse(results_bias["position_error"])[1],
            compute_rmse(results_with_correction["position_error"])[1],
            compute_rmse(results_slip["position_error"])[1],
        ],
        "rmse_z_m": [
            compute_rmse(results_nominal["position_error"])[2],
            compute_rmse(results_noisy["position_error"])[2],
            compute_rmse(results_bias["position_error"])[2],
            compute_rmse(results_with_correction["position_error"])[2],
            compute_rmse(results_slip["position_error"])[2],
        ],
    }
)
metric_table

### Task 8 — Tune EKF process noise and measurement noise

**Practical task:** Tune the EKF noise parameters and observe how they affect the estimate.

In this task, IMU noise and foot slip are enabled, so the estimator must balance noisy prediction and imperfect measurements. Modify the process noise (`PROCESS_NOISE_*`) and measurement noise (`MEASUREMENT_NOISE_FOOT`, `MEASUREMENT_NOISE_SCALE`), then regenerate the dataset and run the EKF.

After each change, compare the RMSE and trajectory plots.

**💡 Hint:** Try increasing and decreasing each parameter separately. Observe how the filter behavior changes when it trusts the IMU more versus when it trusts the foot contacts more.

**Questions:**

1. What do process noise and measurement noise represent?  
2. What happens if process noise is too small?  
3. What happens if measurement noise is too large?  

**Answers:**

1. TODO
2. TODO
3. TODO

In [ ]:
# Add IMU noise
USE_IMU_NOISE = True
ACC_NOISE_STD = 0.1                               
GYRO_NOISE_STD = np.array([0.001, 0.001, 0.005])      

# Add foot slip
FOOT_SLIP_STD = 0.01                             
FOOT_SLIP_ENABLE_Z = True 

# EKF Process noise (IMU)
PROCESS_NOISE_ROT = 5e-2                              #TODO
PROCESS_NOISE_VEL = np.array([7e-2, 3e-2, 2e-2])      #TODO
PROCESS_NOISE_POS = np.array([7e-2, 3e-2, 2e-2])      #TODO

# EKF Measurement noise (foot updates)
MEASUREMENT_NOISE_FOOT = np.array([7e-3, 3e-3, 2e-3]) #TODO
MEASUREMENT_NOISE_SCALE = 2.0                         #TODO

generate_circular_trajectory_csv(overwrite_existing_file=True)
dataset_noisy = load_selected_dataset()
results_tuned = run_ekf_on_dataset(dataset_noisy, disable_foot_updates=False)

print("EKF tuning simulation run complete.")
print("Position RMSE [x, y, z] (m):", np.round(compute_rmse(results_tuned["position_error"]), 4))

plot_trajectory_2d(dataset["ground_truth_position"][1:], results_tuned["estimated_position"][1:], 1, "Top View (XY)", "Y (m)")
plot_trajectory_2d(dataset["ground_truth_position"][1:], results_tuned["estimated_position"][1:], 2, "Side View (XZ)", "Z (m)")

### Task 9 — Visualize the uncertainty

**Practical task:** Visualize the estimator uncertainty using `plot_xy_trajectory_with_covariance_ellipses(...)` and observe how the ellipses evolve along the trajectory.

**💡 Hint:** Use `XY_COVARIANCE_ELLIPSE_STRIDE = 29` to obtain a clean plot with one ellipse per step and a smooth variation across gait phases. This way, each ellipse corresponds roughly to one step. The smooth variation in size comes from a slight phase shift between the gait cycle (~30 samples) and the plotting stride (29), allowing different phases of the step to be visualized along the trajectory.


**Questions:**

1. What does the size of the ellipses represent?  
2. When do the ellipses grow or shrink?  
3. What happens to the uncertainty when no feet are in contact with the ground?

**Answers:**

1. TODO
2. TODO
3. TODO

In [ ]:
XY_COVARIANCE_ELLIPSE_STRIDE = 29  # ~step length (30) → slight phase shift (-1) per step → smooth spread of ellipses

plot_xy_trajectory_with_covariance_ellipses(
    dataset_noisy["ground_truth_position"][1:],
    results_tuned["estimated_position"][1:],
    results_tuned["covariance_history"][1:],
    ellipse_stride=XY_COVARIANCE_ELLIPSE_STRIDE,
    max_samples=500,  # ≈ one full circle (~16–17 steps)
)

### Task 10 — Apply the filter to the real dataset and tune

A video of the Unitree Go2 quadruped walking in a circular trajectory in an indoor scenario, tracked by a Vicon motion capture system, is shown below. The corresponding dataset (index L3), including IMU measurements, foot positions, and contact states, is provided as a CSV file in this notebook. The full dataset is available on Zenodo: https://zenodo.org/records/19336009

In [ ]:
from IPython.display import Video, display
display(Video("./figures/quadruped_walking_exp.mp4", embed=True, width=640, height=480))

**Practical task:** switch to the real dataset only after understanding the simulation.
Start from the same EKF, then tune process and measurement noise.

**Questions:**

1. Why is real-world estimation harder?
2. What assumptions does the EKF rely on?
3. How would you improve this system for rough terrain?

**Answers:**

1. TODO
2. TODO
3. TODO

In [ ]:
USE_REAL_DATA = ...                                   # TODO

# EKF Process noise (IMU)
PROCESS_NOISE_ROT = 5e-2                              #TODO
PROCESS_NOISE_VEL = np.array([7e-2, 3e-2, 2e-2])      #TODO
PROCESS_NOISE_POS = np.array([7e-2, 3e-2, 2e-2])      #TODO

# EKF Measurement noise (foot updates)
MEASUREMENT_NOISE_FOOT = np.array([7e-3, 3e-3, 2e-3]) #TODO
MEASUREMENT_NOISE_SCALE = 2.0                         #TODO

real_dataset = load_selected_dataset(USE_REAL_DATA)
aligned_baseline_position = align_baseline_if_available(real_dataset, USE_REAL_DATA)
real_results = run_ekf_on_dataset(real_dataset, disable_foot_updates=False)

print("Real data run complete.")
print_summary(
    real_dataset["ground_truth_position"],
    real_dataset["time_s"],
    real_results["position_error"],
    real_results["rotation_error_vectors"],
)
    
plot_trajectory_2d(real_dataset["ground_truth_position"][1:], real_results["estimated_position"][1:], 1, "Top View (XY)", "Y (m)", aligned_baseline_position[1:] if aligned_baseline_position is not None else None,)
plot_trajectory_2d(real_dataset["ground_truth_position"][1:], real_results["estimated_position"][1:], 2, "Side View (XZ)", "Z (m)", aligned_baseline_position[1:] if aligned_baseline_position is not None else None,)